# Data Profiling and Visualization

This notebook profiles the files in the data folder, generates summary tables, and saves charts to PNG files.

In [1]:
import json
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')

base_dir = Path.cwd()
data_dir = base_dir / 'data'
output_dir = base_dir / 'outputs'
fig_dir = output_dir / 'figures'
output_dir.mkdir(exist_ok=True)
fig_dir.mkdir(exist_ok=True)

print('Workspace:', base_dir)
print('Data directory:', data_dir)

Workspace: c:\Users\91994\Downloads\Capstone_project
Data directory: c:\Users\91994\Downloads\Capstone_project\data


In [2]:
def load_dataframes():
    frames = {}
    frames['customers.csv'] = pd.read_csv(data_dir / 'customers.csv')
    frames['products.csv'] = pd.read_csv(data_dir / 'products.csv')
    for name in ['orders_day1.json', 'orders_day2.json']:
        with open(data_dir / name, 'r', encoding='utf-8') as f:
            frames[name] = pd.DataFrame(json.load(f))
    for name in ['order_items_day1.json', 'order_items_day2.json']:
        with open(data_dir / name, 'r', encoding='utf-8') as f:
            frames[name] = pd.DataFrame(json.load(f))
    for name in ['clickstream_day1.json', 'clickstream_day2.json']:
        with open(data_dir / name, 'r', encoding='utf-8') as f:
            frames[name] = pd.DataFrame(json.load(f))
    return frames

frames = load_dataframes()
for name, df in frames.items():
    print(name, df.shape)

customers.csv (5000, 3)
products.csv (800, 4)
orders_day1.json (20000, 5)
orders_day2.json (4000, 6)
order_items_day1.json (55000, 5)
order_items_day2.json (11000, 6)
clickstream_day1.json (15000, 3)
clickstream_day2.json (15000, 3)


In [3]:
# Standardize dtypes for downstream analytics
orders_df = pd.concat([frames['orders_day1.json'], frames['orders_day2.json']], ignore_index=True)
order_items_df = pd.concat([frames['order_items_day1.json'], frames['order_items_day2.json']], ignore_index=True)
products_df = frames['products.csv']
customers_df = frames['customers.csv']
clickstream_df = pd.concat([frames['clickstream_day1.json'], frames['clickstream_day2.json']], ignore_index=True)

orders_df['order_ts'] = pd.to_datetime(orders_df['order_ts'], errors='coerce')
order_items_df['quantity'] = pd.to_numeric(order_items_df['quantity'], errors='coerce')
order_items_df['unit_price'] = pd.to_numeric(order_items_df['unit_price'], errors='coerce')
order_items_df['line_total'] = pd.to_numeric(order_items_df['line_total'], errors='coerce')
products_df['unit_price'] = pd.to_numeric(products_df['unit_price'], errors='coerce')
products_df['active_flag'] = pd.to_numeric(products_df['active_flag'], errors='coerce')

combined_frames = {
    'customers.csv': customers_df,
    'products.csv': products_df,
    'orders_day1.json': frames['orders_day1.json'],
    'orders_day2.json': frames['orders_day2.json'],
    'order_items_day1.json': frames['order_items_day1.json'],
    'order_items_day2.json': frames['order_items_day2.json'],
    'clickstream_day1.json': frames['clickstream_day1.json'],
    'clickstream_day2.json': frames['clickstream_day2.json'],
}

def build_profile_summary(df, file_name):
    rows = []
    for column in df.columns:
        series = df[column]
        if pd.api.types.is_numeric_dtype(series):
            range_text = f"{series.min()} to {series.max()}"
        elif pd.api.types.is_datetime64_any_dtype(series):
            range_text = f"{series.min()} to {series.max()}"
        else:
            range_text = f"{series.nunique()} unique values"
        rows.append({
            'file_name': file_name,
            'column_name': column,
            'null_rate_pct': round(series.isna().mean() * 100, 2),
            'dtype': str(series.dtype),
            'value_range': range_text
        })
    return pd.DataFrame(rows)

column_profile_rows = []
file_profile_rows = []
for file_name, df in combined_frames.items():
    profile_df = build_profile_summary(df, file_name)
    column_profile_rows.append(profile_df)
    file_profile_rows.append({
        'file_name': file_name,
        'rows': len(df),
        'columns': len(df.columns),
        'duplicate_rows': int(df.duplicated().sum()),
        'null_column_count': int(df.isna().any().sum())
    })

column_profile_df = pd.concat(column_profile_rows, ignore_index=True)
file_profile_df = pd.DataFrame(file_profile_rows)

column_profile_df.to_csv(output_dir / 'column_profile_summary.csv', index=False)
file_profile_df.to_csv(output_dir / 'file_profile_summary.csv', index=False)

display(file_profile_df)
display(column_profile_df.head(20))

,file_name,rows,columns,duplicate_rows,null_column_count
0,customers.csv,5000,3,0,1
1,products.csv,800,4,0,0
2,orders_day1.json,20000,5,0,0
3,orders_day2.json,4000,6,0,1
4,order_items_day1.json,55000,5,24,0
5,order_items_day2.json,11000,6,0,1
6,clickstream_day1.json,15000,3,0,0
7,clickstream_day2.json,15000,3,0,0


,file_name,column_name,null_rate_pct,dtype,value_range
0,customers.csv,customer_id,0.00,str,4950 unique values
1,customers.csv,customer_name,0.00,str,5000 unique values
2,customers.csv,email,1.00,str,4950 unique values
3,products.csv,product_id,0.00,str,800 unique values
4,products.csv,category,0.00,str,4 unique values
5,products.csv,unit_price,0.00,float64,5.36 to 499.04
6,products.csv,active_flag,0.00,int64,0 to 1
7,orders_day1.json,order_id,0.00,str,20000 unique values
8,orders_day1.json,customer_id,0.00,str,4860 unique values
9,orders_day1.json,order_ts,0.00,str,20000 unique values


In [4]:
# 1) Order volume by day
orders_daily = orders_df.set_index('order_ts').resample('D').size().reset_index(name='order_count')
orders_daily.columns = ['order_date', 'order_count']
plt.figure(figsize=(10, 4))
sns.lineplot(data=orders_daily, x='order_date', y='order_count', marker='o')
plt.title('Order Volume by Day')
plt.xlabel('Date')
plt.ylabel('Orders')
plt.tight_layout()
plt.savefig(fig_dir / 'order_volume_by_day.png', dpi=300)
plt.close()

# 2) Revenue distribution
plt.figure(figsize=(10, 4))
sns.histplot(order_items_df['line_total'].dropna(), bins=30, kde=True)
plt.title('Revenue Distribution')
plt.xlabel('Line Total')
plt.ylabel('Frequency')
plt.tight_layout()
plt.savefig(fig_dir / 'revenue_distribution.png', dpi=300)
plt.close()

# 3) Top 10 categories by revenue
category_revenue = (
    order_items_df.merge(products_df[['product_id', 'category']], on='product_id', how='left')
    .assign(category=lambda x: x['category'].fillna('Unknown'))
    .groupby('category')['line_total'].sum()
    .sort_values(ascending=False)
    .head(10)
)
plt.figure(figsize=(10, 4))
sns.barplot(x=category_revenue.values, y=category_revenue.index, orient='h', palette='viridis')
plt.title('Top 10 Categories by Revenue')
plt.xlabel('Revenue')
plt.ylabel('Category')
plt.tight_layout()
plt.savefig(fig_dir / 'top_categories_by_revenue.png', dpi=300)
plt.close()

# 4) Null-rate heatmap across all files
null_heatmap_df = (
    column_profile_df.pivot(index='file_name', columns='column_name', values='null_rate_pct')
    .fillna(0)
)
plt.figure(figsize=(14, 6))
sns.heatmap(null_heatmap_df, cmap='YlGnBu', annot=True, fmt='.1f')
plt.title('Null-rate Heatmap Across Files')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(fig_dir / 'null_rate_heatmap.png', dpi=300)
plt.close()

print('Charts saved to', fig_dir)

C:\Users\91994\AppData\Local\Temp\ipykernel_10660\1855527584.py:32: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=category_revenue.values, y=category_revenue.index, orient='h', palette='viridis')


Charts saved to c:\Users\91994\Downloads\Capstone_project\outputs\figures


Generated outputs are saved in the outputs folder, and the charts are exported as PNG files in the figures subfolder.